# Claude Agent SDK - Python Examples

This notebook demonstrates various examples of using the Claude Agent SDK in Python. Each example builds on the previous one, showcasing different capabilities and patterns.

## Prerequisites

Before running these examples, ensure you have:

1. Installed the Claude Agent SDK:
   ```bash
   pip install claude-agent-sdk
   ```

2. Set your API key:
   ```bash
   export ANTHROPIC_API_KEY='your-api-key-here'
   ```

3. Created a working directory structure:
   ```bash
   mkdir -p agent/custom_scripts
   ```

## Example 1: Hello World - Basic Query

The simplest way to use the Claude Agent SDK is with the `query()` function. This creates a one-shot interaction where you send a prompt and process the response.

### Key Concepts:
- **`query()`**: Main entry point for single-turn interactions
- **Async iteration**: The SDK uses async generators to stream responses
- **Message types**: Different message types (assistant, tool_use, result) indicate different parts of the interaction
- **Options**: Configure behavior like max turns, working directory, model selection, and allowed tools

### What This Example Does:
1. Sends a simple prompt to Claude
2. Streams the response
3. Extracts and prints the text content

In [ ]:
import asyncio
import os
from pathlib import Path
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions

async def hello_world():
    """Basic query example - single turn conversation."""
    
    # Configure the agent
    options = ClaudeAgentOptions(
        max_turns=10,
        cwd=Path.cwd() / "agent",
        model="opus",
        allowed_tools=[
            "Task", "Bash", "Glob", "Grep", "Read", "Edit", 
            "Write", "NotebookEdit", "WebFetch", "TodoWrite", 
            "WebSearch"
        ]
    )
    
    # Create client and send query
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt='Hello, Claude! Please introduce yourself in one sentence.')
        
        # Stream and process the response
        async for msg in client.receive_response():
            if type(msg).__name__ == 'AssistantMessage':
                # Extract text content from the message
                for content in msg.message.content:
                    if content.type == 'text':
                        print(f'Claude says: {content.text}')

# Run the example
await hello_world()

## Example 2: Pre-Tool Use Hooks

Hooks allow you to intercept and control the agent's behavior. They're useful for:
- Validating tool usage
- Enforcing security policies
- Logging and monitoring
- Modifying tool inputs

### Key Concepts:
- **PreToolUse hooks**: Execute before a tool is called
- **HookMatcher**: Pattern matching to target specific tools
- **Block decision**: Hooks can block tool execution and provide feedback
- **HookJSONOutput**: Return type with `continue`, `decision`, and `stopReason` fields

### What This Example Does:
1. Enforces that JavaScript/TypeScript files must be written to `custom_scripts/`
2. Blocks write operations to other directories
3. Provides clear feedback to the agent when blocked

In [ ]:
from claude_agent_sdk import HookMatcher, HookJSONOutput
from typing import Any, Dict

async def validate_script_location(hook_input: Dict[str, Any]) -> HookJSONOutput:
    """Hook to enforce script files are written to custom_scripts directory."""
    
    tool_name = hook_input['tool_name']
    tool_input = hook_input['tool_input']
    
    # Only check Write/Edit tools
    if tool_name not in ['Write', 'Edit', 'MultiEdit']:
        return HookJSONOutput(continue_execution=True)
    
    # Get file path from tool input
    file_path = tool_input.get('file_path', '')
    
    # Check if it's a JS/TS file
    if file_path.endswith('.js') or file_path.endswith('.ts'):
        custom_scripts_path = Path.cwd() / 'agent' / 'custom_scripts'
        file_path_obj = Path(file_path)
        
        # Block if not in custom_scripts
        if not str(file_path_obj).startswith(str(custom_scripts_path)):
            return HookJSONOutput(
                decision='block',
                stop_reason=(
                    f"Script files (.js and .ts) must be written to the custom_scripts directory. "
                    f"Please use the path: {custom_scripts_path / file_path_obj.name}"
                ),
                continue_execution=False
            )
    
    return HookJSONOutput(continue_execution=True)


async def hello_world_with_hooks():
    """Hello world example with security hooks."""
    
    # Define hooks
    hooks = {
        'PreToolUse': [
            HookMatcher(
                matcher="Write|Edit|MultiEdit",
                hooks=[validate_script_location]
            )
        ]
    }
    
    options = ClaudeAgentOptions(
        max_turns=10,
        cwd=Path.cwd() / "agent",
        model="opus",
        allowed_tools=["Write", "Read", "Edit"],
        hooks=hooks
    )
    
    async with ClaudeSDKClient(options=options) as client:
        # This will trigger our hook if Claude tries to write a .js/.ts file
        await client.query(
            prompt='Create a simple hello.js file that prints "Hello World"'
        )
        
        async for msg in client.receive_response():
            if type(msg).__name__ == 'AssistantMessage':
                for content in msg.message.content:
                    if content.type == 'text':
                        print(content.text)

# Run the example
await hello_world_with_hooks()

## Example 3: Multi-Turn Conversations

The Claude Agent SDK supports multi-turn conversations where context is preserved across multiple interactions. This is essential for:
- Building chatbots
- Complex workflows that require multiple steps
- Conversations where Claude needs to remember previous interactions

### Key Concepts:
- **Context preservation**: Each query builds on previous ones
- **Multiple `query()` calls**: Within the same client session
- **Session state**: Automatically managed by the SDK

### What This Example Does:
1. Asks Claude to calculate 5 + 3
2. In a follow-up turn, asks to multiply "that" by 2
3. Demonstrates that Claude remembers the previous answer (8)

In [ ]:
async def multi_turn_conversation():
    """Demonstrate multi-turn conversation with context preservation."""
    
    print("=== Multi-Turn Conversation ===")
    print()
    
    options = ClaudeAgentOptions(
        model="sonnet",
        max_turns=10
    )
    
    async with ClaudeSDKClient(options=options) as client:
        # Turn 1: Initial question
        print("Turn 1: What is 5 + 3?")
        await client.query(prompt='What is 5 + 3? Just give me the number.')
        
        async for msg in client.receive_response():
            if type(msg).__name__ == 'AssistantMessage':
                for content in msg.message.content:
                    if content.type == 'text':
                        print(f"Claude: {content.text}")
                        print()
        
        # Turn 2: Follow-up that requires context from Turn 1
        print("Turn 2: Multiply that by 2")
        await client.query(prompt='Multiply that by 2. Just give me the number.')
        
        async for msg in client.receive_response():
            if type(msg).__name__ == 'AssistantMessage':
                for content in msg.message.content:
                    if content.type == 'text':
                        print(f"Claude: {content.text}")
                        print("(Claude remembered that the previous answer was 8!)")

# Run the example
await multi_turn_conversation()

## Example 4: Resume Generator

This is a practical example that combines multiple features:
- Web search for gathering information
- System prompts for guiding agent behavior
- File operations for saving output
- Tool restrictions for focused tasks

### Key Concepts:
- **System prompts**: Guide the agent's behavior and workflow
- **Tool selection**: Limit tools to only what's needed
- **Web search**: Use WebSearch tool to gather information
- **Structured output**: Create formatted documents programmatically

### What This Example Does:
1. Takes a person's name as input
2. Uses web search to research their background
3. Generates a professional 1-page resume
4. Saves it as a .docx file

In [ ]:
async def generate_resume(person_name: str):
    """Generate a professional resume by researching a person online."""
    
    print(f"\n📝 Generating resume for: {person_name}")
    print("=" * 50)
    
    # Define the workflow for the agent
    system_prompt = """You are a professional resume writer. Research a person and create a 1-page .docx resume.

WORKFLOW:
1. WebSearch for the person's background (LinkedIn, GitHub, company pages)
2. Create a .docx file using the docx library

OUTPUT:
- Script: agent/custom_scripts/generate_resume.py
- Resume: agent/custom_scripts/resume.docx

PAGE FIT (must be exactly 1 page):
- 0.5 inch margins, Name 24pt, Headers 12pt, Body 10pt
- 2-3 bullet points per job, ~80-100 chars each
- Max 3 job roles, 2-line summary, 2-line skills"""
    
    # Ensure output directory exists
    output_dir = Path.cwd() / 'agent' / 'custom_scripts'
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Configure the agent with specific tools for this task
    options = ClaudeAgentOptions(
        max_turns=30,
        cwd=Path.cwd(),
        model='sonnet',
        allowed_tools=['Skill', 'WebSearch', 'WebFetch', 'Bash', 'Write', 'Read', 'Glob'],
        setting_sources=['project'],  # Load skills from .claude/skills/
        system_prompt=system_prompt
    )
    
    prompt = f'Research "{person_name}" and create a professional 1-page resume as a .docx file. Search for their professional background, experience, education, and skills.'
    
    print("\n🔍 Researching and creating resume...\n")
    
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt=prompt)
        
        async for msg in client.receive_response():
            if type(msg).__name__ == 'AssistantMessage':
                for content in msg.message.content:
                    if content.type == 'text':
                        print(content.text)
                    elif content.type == 'tool_use':
                        if content.name == 'WebSearch':
                            query = content.input.get('query', '')
                            print(f"\n🔍 Searching: \"{query}\"")
                        else:
                            print(f"\n🔧 Using tool: {content.name}")
    
    # Check if resume was created
    expected_path = output_dir / 'resume.docx'
    if expected_path.exists():
        print("\n" + "=" * 50)
        print(f"📄 Resume saved to: {expected_path}")
        print("=" * 50)
    else:
        print("\n❌ Resume file was not created. Check the output above for errors.")

# Run the example
# await generate_resume("Jane Doe")  # Uncomment and replace with actual name

## Example 5: Multi-Agent System with Subagents

The most powerful pattern in the Claude Agent SDK is using multiple specialized agents that work together. This is ideal for:
- Complex workflows with distinct phases
- Separating concerns (research vs. analysis vs. reporting)
- Optimizing costs (use cheaper models for simpler tasks)
- Parallel processing of different aspects

### Key Concepts:
- **AgentDefinition**: Define specialized subagents with specific tools and prompts
- **Task tool**: The lead agent uses the Task tool to delegate to subagents
- **Specialization**: Each subagent has focused responsibilities and limited tools
- **Coordination**: The lead agent orchestrates the workflow

### Architecture:
```
Lead Agent (orchestrator)
    ├── Researcher (web search, information gathering)
    ├── Data Analyst (metrics extraction, visualization)
    └── Report Writer (PDF generation, synthesis)
```

### What This Example Does:
1. Sets up a lead agent that coordinates three specialized subagents
2. **Researcher**: Uses web search to gather information
3. **Data Analyst**: Extracts metrics and creates charts with matplotlib
4. **Report Writer**: Synthesizes everything into a professional PDF report

In [ ]:
from claude_agent_sdk import AgentDefinition
from datetime import datetime

async def research_agent_example():
    """Multi-agent research system with specialized subagents."""
    
    # Create session directory for outputs
    session_dir = Path.cwd() / "sessions" / datetime.now().strftime("%Y%m%d_%H%M%S")
    session_dir.mkdir(parents=True, exist_ok=True)
    
    # Create output directories
    (session_dir / "files" / "research_notes").mkdir(parents=True, exist_ok=True)
    (session_dir / "files" / "charts").mkdir(parents=True, exist_ok=True)
    (session_dir / "files" / "reports").mkdir(parents=True, exist_ok=True)
    (session_dir / "files" / "data").mkdir(parents=True, exist_ok=True)
    
    # Define the lead agent's system prompt
    lead_agent_prompt = """You are a research orchestrator that coordinates specialized subagents.

WORKFLOW:
1. Use the 'researcher' agent to gather information via web search
2. Use the 'data-analyst' agent to extract metrics and create visualizations
3. Use the 'report-writer' agent to synthesize everything into a PDF report

Always delegate to the appropriate subagent using the Task tool."""
    
    # Define specialized subagents
    agents = {
        "researcher": AgentDefinition(
            description=(
                "Use this agent when you need to gather research information on any topic. "
                "The researcher uses web search to find relevant information, articles, and sources "
                "from across the internet. Writes research findings to files/research_notes/ "
                "for later use by report writers. Ideal for complex research tasks "
                "that require deep searching and cross-referencing."
            ),
            tools=["WebSearch", "Write"],
            prompt="""You are a research specialist. Use web search to thoroughly research the given topic.
            
Write your findings to files/research_notes/ with clear, organized notes including:
- Key facts and statistics
- Important trends and developments
- Notable quotes and sources
- Relevant context and background

Be thorough and cite your sources.""",
            model="haiku"
        ),
        
        "data-analyst": AgentDefinition(
            description=(
                "Use this agent AFTER researchers have completed their work to generate quantitative "
                "analysis and visualizations. The data-analyst reads research notes from files/research_notes/, "
                "extracts numerical data (percentages, rankings, trends, comparisons), and generates "
                "charts using Python/matplotlib via Bash. Saves charts to files/charts/ and writes "
                "a data summary to files/data/. Use this before the report-writer to add visual insights."
            ),
            tools=["Glob", "Read", "Bash", "Write"],
            prompt="""You are a data analyst. Read research notes and extract quantitative insights.

TASKS:
1. Read files from files/research_notes/
2. Extract numerical data (percentages, trends, comparisons)
3. Create visualizations using matplotlib via Bash
4. Save charts to files/charts/ as PNG files
5. Write a data summary to files/data/

Use clear, professional chart formatting with titles, labels, and legends.""",
            model="haiku"
        ),
        
        "report-writer": AgentDefinition(
            description=(
                "Use this agent when you need to create a formal research report document. "
                "The report-writer reads research findings from files/research_notes/, data analysis "
                "from files/data/, and charts from files/charts/, then synthesizes them into clear, "
                "concise, professionally formatted PDF reports in files/reports/ using reportlab. "
                "Ideal for creating structured documents with proper citations, data, and embedded visuals. "
                "Does NOT conduct web searches - only reads existing research notes and creates PDF reports."
            ),
            tools=["Skill", "Write", "Glob", "Read", "Bash"],
            prompt="""You are a report writer. Synthesize research and data into a professional PDF.

TASKS:
1. Read research notes from files/research_notes/
2. Read data summaries from files/data/
3. Identify charts in files/charts/
4. Create a well-structured PDF report with:
   - Executive summary
   - Key findings with data
   - Embedded charts and visualizations
   - Clear sections and formatting
5. Save to files/reports/

Use professional formatting and cite sources from research notes.""",
            model="haiku"
        )
    }
    
    # Configure the lead agent
    options = ClaudeAgentOptions(
        permission_mode="bypassPermissions",
        setting_sources=["project"],
        system_prompt=lead_agent_prompt,
        allowed_tools=["Task"],  # Lead agent only uses Task to delegate
        agents=agents,
        model="haiku",
        cwd=session_dir
    )
    
    print("\n" + "=" * 50)
    print("  Multi-Agent Research System")
    print("=" * 50)
    print("\nThis system uses three specialized agents:")
    print("  1. Researcher - Gathers information via web search")
    print("  2. Data Analyst - Extracts metrics and creates charts")
    print("  3. Report Writer - Synthesizes into a PDF report")
    print(f"\nSession directory: {session_dir}")
    print()
    
    # Example research topic
    topic = "The impact of artificial intelligence on software development in 2024"
    
    async with ClaudeSDKClient(options=options) as client:
        await client.query(
            prompt=f"Research the following topic and create a comprehensive PDF report: {topic}"
        )
        
        async for msg in client.receive_response():
            if type(msg).__name__ == 'AssistantMessage':
                for content in msg.message.content:
                    if content.type == 'text':
                        print(content.text)
                    elif content.type == 'tool_use':
                        if content.name == 'Task':
                            subagent = content.input.get('subagent_type', 'unknown')
                            print(f"\n🤖 Delegating to: {subagent}")
    
    print(f"\n✅ Research complete! Check {session_dir} for outputs.")

# Run the example
# await research_agent_example()  # Uncomment to run

## Example 6: Interactive Chat with Session Management

For building interactive applications, you'll want to manage sessions and handle real-time streaming. This example shows a complete interactive chat loop.

### Key Concepts:
- **Interactive loop**: Continuously accept user input
- **Session persistence**: Maintain conversation history
- **Transcript logging**: Save conversations to disk
- **Tool tracking**: Monitor which tools are being used

### What This Example Does:
1. Creates an interactive REPL-style interface
2. Maintains conversation history across turns
3. Logs all interactions to a transcript file
4. Tracks tool usage for monitoring

In [ ]:
async def interactive_chat():
    """Interactive chat loop with session management and logging."""
    
    # Create session directory
    session_dir = Path.cwd() / "sessions" / datetime.now().strftime("%Y%m%d_%H%M%S")
    session_dir.mkdir(parents=True, exist_ok=True)
    transcript_file = session_dir / "transcript.txt"
    
    # Configure the agent
    options = ClaudeAgentOptions(
        model="sonnet",
        max_turns=100,
        allowed_tools=["Bash", "Read", "Write", "Glob", "Grep", "WebSearch", "WebFetch"],
        cwd=Path.cwd()
    )
    
    print("\n" + "=" * 50)
    print("  Interactive Chat with Claude")
    print("=" * 50)
    print("\nType 'exit' or 'quit' to end the conversation.")
    print(f"Session logs: {session_dir}")
    print()
    
    try:
        async with ClaudeSDKClient(options=options) as client:
            # Open transcript file
            with open(transcript_file, 'w', encoding='utf-8') as transcript:
                transcript.write(f"Session started: {datetime.now()}\n\n")
                
                while True:
                    # Get user input
                    try:
                        user_input = input("You: ").strip()
                    except (EOFError, KeyboardInterrupt):
                        print("\nExiting...")
                        break
                    
                    if not user_input or user_input.lower() in ['exit', 'quit', 'q']:
                        print("Goodbye!")
                        break
                    
                    # Log user input
                    transcript.write(f"You: {user_input}\n")
                    
                    # Send to agent
                    await client.query(prompt=user_input)
                    
                    print("Claude: ", end="", flush=True)
                    transcript.write("Claude: ")
                    
                    # Stream response
                    async for msg in client.receive_response():
                        if type(msg).__name__ == 'AssistantMessage':
                            for content in msg.message.content:
                                if content.type == 'text':
                                    print(content.text, end="", flush=True)
                                    transcript.write(content.text)
                                elif content.type == 'tool_use':
                                    tool_name = content.name
                                    print(f"\n[Using tool: {tool_name}]", flush=True)
                                    transcript.write(f" [tool: {tool_name}] ")
                    
                    print()  # New line after response
                    transcript.write("\n\n")
                    transcript.flush()
                
                transcript.write(f"\nSession ended: {datetime.now()}\n")
    
    except Exception as e:
        print(f"\nError: {e}")
    
    print(f"\nTranscript saved to: {transcript_file}")

# Run the example
# await interactive_chat()  # Uncomment to run

## Key Patterns and Best Practices

### 1. Model Selection
- **Opus**: Most capable, use for complex reasoning and planning
- **Sonnet**: Balanced performance and cost, good default choice
- **Haiku**: Fast and economical, ideal for simple tasks and subagents

### 2. Tool Management
- Use `allowed_tools` to limit agent capabilities
- Give subagents minimal tools needed for their specific task
- Lead agents often only need the `Task` tool for delegation

### 3. System Prompts
- Define clear workflows and expectations
- Specify output formats and locations
- Include examples when behavior is critical
- Keep prompts focused on the agent's specific role

### 4. Hook Usage
- **PreToolUse**: Validate inputs, enforce policies, add context
- **PostToolUse**: Transform outputs, log activity, trigger follow-ups
- Use HookMatcher patterns to target specific tools
- Return clear `stopReason` messages when blocking

### 5. Multi-Agent Architecture
- Separate concerns: research, analysis, synthesis
- Use file system for inter-agent communication
- Define clear handoff points between agents
- Consider using cheaper models for specialized subagents

### 6. Error Handling
- Wrap agent calls in try-except blocks
- Use `async with` for automatic cleanup
- Log errors to session directories
- Provide graceful fallbacks

### 7. Cost Optimization
- Use Haiku for simple, focused tasks
- Limit `max_turns` appropriately
- Restrict tools to only what's needed
- Cache frequently accessed resources

### 8. Session Management
- Create unique session directories with timestamps
- Save transcripts for debugging and auditing
- Track tool usage for monitoring
- Clean up old sessions periodically

## Common Use Cases

### Chatbots and Assistants
Use the interactive chat pattern with multi-turn conversations. Add custom tools for domain-specific actions.

### Research and Analysis
Use the multi-agent pattern with specialized researchers, analysts, and report writers. Great for comprehensive investigations.

### Code Generation
Provide file system tools and clear system prompts. Use hooks to enforce code style and security policies.

### Document Processing
Use Read/Write tools with structured prompts. Consider specialized agents for different document types.

### Data Pipelines
Chain multiple specialized agents together. Use the file system as a data bus between agents.

### Automation Workflows
Combine Bash, Web, and File tools. Use hooks to add safety checks and logging.

## Debugging Tips

### Enable verbose logging
```python
import logging
logging.basicConfig(level=logging.DEBUG)
```

### Inspect message types
```python
async for msg in client.receive_response():
    print(f"Message type: {type(msg).__name__}")
    print(f"Message content: {msg}")
```

### Save all tool calls
Use PostToolUse hooks to log every tool invocation and result to a file.

### Test with Haiku first
Haiku is faster and cheaper for debugging workflow issues before scaling to Opus.

### Check session directories
Always verify your working directories are correct and files are being written where expected.

## Next Steps

1. **Explore the examples**: Run each example above and modify them for your needs
2. **Read the docs**: Check out the full Claude Agent SDK documentation
3. **Build custom tools**: Create domain-specific tools for your use case
4. **Design your architecture**: Plan your agent system before coding
5. **Start simple**: Begin with single-agent systems and add complexity as needed
6. **Monitor costs**: Track API usage and optimize with appropriate model selection

## Resources

- Claude Agent SDK GitHub: https://github.com/anthropics/claude-agent-sdk
- API Documentation: https://docs.anthropic.com/
- More Examples: See the `claude-agent-sdk-demos` repository